# Comparing vacuum equilibrium to Poincaré

 We want to investigate the difference of an equilibrium solution to the actual vacuum ($I_{tor}=0, p=0$) solution obtained via fieldline tracing. For geometries beyond cylindrical coordinates, however, the surface normals and their resulting Poincaré sections might not be ideal; They can produce unnecessarily elongated cross-sections. Thus, we create cross-sections from a GVEC `state` object, that are aligned with the frame of our equilibrium solution via the utility function `intersection_planes_from_state`. To showcase this, we use a [quasr-case](https://quasr.flatironinstitute.org/model/2265187) from the vast and excellent [quasr-database](https://quasr.flatironinstitute.org/)

In [ ]:
# set the number of OpenMP threads before importing gvec
import os

os.environ["OMP_NUM_THREADS"] = "2"

import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from pathlib import Path
from gvec import Run
from gvec.coils import intersection_planes_from_state, trace_fieldlines, get_phi_edge_from_coils
from gvec.scripts.quasr import main as load_quasr, get_coils_from_json_file
from gvec.plotting import plot_zeta_cuts
from gvec.util import chdir, read_parameters

First, we setup the quasr case and run GVEC with the provided last closed flux-surface. Also we extract the coils into a GVEC `CoilSet`.

In [ ]:
# choose a testcase / quasr ID
ID = 2265187

n_rhos = 10

# create a corresponding directory
save_path = Path("quasr")
if not save_path.exists():
    save_path.mkdir()
with chdir(save_path):
    # download th quasr case and translate it into a GVEC input file
    load_quasr([str(ID), "--tol=1e-4"])

    # utility to get the quasr coils into a GVEC CoilSet
    coil_set = get_coils_from_json_file(f"quasr-{ID:07d}.json")

    # run the quasr case with GVEC
    params = read_parameters(f"quasr-{ID:07d}-parameters.toml")

    # increase this tollerance for more accurate equilibrium results
    params["minimize_tol"] = 1e-4

    gvec_run = Run(params, quiet=False)
    # Since we do not know the total toroidal flux we can estimate it from the coil-field
    gvec_run.parameters["PhiEdge"] = get_phi_edge_from_coils(gvec_run.state, coil_set)

    # run GVEC on the quasr cas
    gvec_run.run()

    # choose the relevant cross-sections
    state = gvec_run.state

Next we can visualize the 3D geometry:

In [ ]:
ev = state.evaluate(
    "pos",
    "mod_B",
    rho=1.0,
    theta=np.linspace(0, 2 * np.pi, 128),
    zeta=np.linspace(0, 2 * np.pi, 64),
).sel(rho=1.0)

boundary = ev.pos
ax3d = coil_set.plot(
    show=False,
    line={"color": "black"},
    showlegend=False,
)
ax3d["layout"]["scene"]["aspectmode"] = "data"
ax3d.add_trace(
    go.Surface(
        x=boundary.sel(xyz="x"),
        y=boundary.sel(xyz="y"),
        z=boundary.sel(xyz="z"),
        surfacecolor=ev.mod_B,
        colorbar_title_text="B / T",
    )
)

As a second part, we initialize fieldline tracing from the equilibrium geometry; that is, we provide points on different flux surfaces as initial conditions for the tracing.

In [ ]:
zetas = np.linspace(0, 2 * np.pi / state.nfp, 4, endpoint=False)

# initialize the fieldlines on the flux-surfaces
rho = np.linspace(0, 0.99, n_rhos)
starts = state.evaluate(
    "pos",
    zeta=0.0,
    theta=0.0,  # np.linspace(0,2*np.pi,5),
    rho=rho,
).pos

# define frame aligned intersection planes
planes = intersection_planes_from_state(state, zetas=zetas)

# perform the fieldline tracing
dt_quasr = trace_fieldlines(
    starts=starts,
    coils=coil_set,
    t=1200,
    n_jobs=min(n_rhos, 2),
    events=planes,
    atol=1e-8,
    rtol=1e-6,
    max_step=0.1,
)

Finally, we can visualize the fieldlines together with the equilibrium flux surfaces. Since we utilized a `State` object to define our intersection planes, the resulting data-tree returned from `trace_fieldlines` also has entries of the type `event_i_X1` and `event_i_X2`, which are the intersection positions of the fieldlines with the i-th specified plane in the corresponding frame-aligned-coordinates X1 and X2. Simply put, they allow us to easily visualize the flux-surfaces and the Poincaré sections together.

In [ ]:
rho = np.linspace(0, 0.99, n_rhos)
theta = np.linspace(0, 2 * np.pi, 128, endpoint=True)
fig, axs = plt.subplots(2, 2, figsize=(8, 8), tight_layout=True)
axs = axs.flatten()
axs = plot_zeta_cuts(state, rho=rho, zeta=zetas, theta=theta, axs=axs, color="k")
for fieldline in dt_quasr:
    ds = dt_quasr[fieldline]
    for i, ax in enumerate(axs):
        ax.scatter(ds[f"event_{i}_X1"], ds[f"event_{i}_X2"], marker=".", s=1, zorder=200)

In this case we see, that the equilibrium field agrees comparatively well with the Poincaré section close to the axis, but that the LCFS is not consistent with the coil-field.

Finally, we can also visualize these fieldlines in 3D.

In [ ]:
ax3Df = coil_set.plot(
    show=False,
    line={"color": "black"},
    showlegend=False,
)
ax3Df["layout"]["scene"]["aspectmode"] = "data"

for fieldline in dt_quasr:
    # plot the fieldlines
    ds = dt_quasr[fieldline]
    ax3Df.add_trace(
        go.Scatter3d(
            x=ds.pos.sel(xyz="x"),
            y=ds.pos.sel(xyz="y"),
            z=ds.pos.sel(xyz="z"),
            mode="lines",
            opacity=0.01,
            line={"color": "green"},
            showlegend=False,
        )
    )

    for i in range(len(zetas)):
        ax3Df.add_trace(
            go.Scatter3d(
                x=np.array(ds[f"event_{i}"].sel(xyz="x")),
                y=np.array(ds[f"event_{i}"].sel(xyz="y")),
                z=np.array(ds[f"event_{i}"].sel(xyz="z")),
                mode="markers",
                marker=dict(size=0.5, color="blue"),
                showlegend=False,
            )
        )
ax3Df.show();